In [1]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np



In [2]:
df = pd.read_csv("../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail()

,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
Date,,,,,,,,,,,
2026-05-08,3.71,3.69,3.74,3.75,3.90,3.92,4.02,4.19,4.38,4.93,4.95
2026-05-11,3.71,3.70,3.77,3.79,3.95,3.96,4.07,4.24,4.42,4.97,4.98
2026-05-12,3.71,3.70,3.77,3.80,4.00,4.01,4.12,4.29,4.46,5.02,5.03
2026-05-13,3.71,3.69,3.77,3.79,3.98,4.00,4.12,4.28,4.46,5.03,5.03
2026-05-14,3.72,3.69,3.76,3.79,4.00,4.04,4.13,4.29,4.47,5.01,5.02


In [ ]:
# Fit to DNS with Ordinary Least Squares

"""
L(t) = 1
S(t) = 1-e^(-xt)/xt
C(t) = S(t) - e^(-xt) 

x = 0.0609
"""

maturities = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
x = 0.0609

# matrix never changes 
# A = np.array([[]])

# for t in maturities:
#     L = 1
#     S = (1 - np.exp(-x * t)) / (x * t)
#     C = S - np.exp(-x * t)

#     np.append(A, [[L, S, C]], axis=0)

A = np.array([
    [
    1,
    (1 - np.exp(-x * t)) / (x * t),
    ((1 - np.exp(-x * t)) / (x * t)) - np.exp(-x * t),
    ]
    for t in maturities
])


rows = []

for row in df.itertuples():
    # get actual yields for each date
    y = [row._1, row._2, row._3, row._4, row._5, row._6, row._7, row._8, row._9, row._10, row._11]

    betas, *_ = np.linalg.lstsq(A, y)
    b1, b2, b3 = betas
    rows.append({'Date': row.Index, 'Beta 1': b1, 'Beta 2': b2, 'Beta 3': b3})

# Dont Rerun file
results_df = pd.DataFrame(rows).set_index("Date")
results_df.to_csv('ns.csv')

finaldf = pd.read_csv("ns.csv")
finaldf.head(5)




C:\Users\thiru\AppData\Local\Temp\ipykernel_41032\611418878.py:40: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  betas, *_ = np.linalg.lstsq(A, y)


,Date,Beta 1,Beta 2,Beta 3
0,2001-07-31,1.335227,2.087442,10.813040
1,2001-08-01,0.997329,2.427643,11.450893
2,2001-08-02,0.520673,2.911515,12.393985
3,2001-08-03,0.294965,3.136327,12.845567
4,2001-08-06,0.378702,3.040047,12.727382
